In [1]:
import xarray as xr
import glob
import pandas as pd
import numpy as np
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
from cartopy.io.shapereader import Reader as ShapeReader

In [2]:
def read_mar_var(path_pattern, var_names, months=[11, 12, 1, 2, 3], time_slice=('19800101', '20220331')):
    """
    Reads MAR NetCDF files, selects specified variables and dimensions,
    filters by months and time range.

    Parameters:
        path_pattern (str): File path pattern for NetCDF files.
        var_names (list): List of variable names to extract.
        months (list, optional): Months to filter. Default is summer in Patagonia.
        time_slice (tuple, optional): Start and end date for filtering. Default is ('19800101', '20220331').

    Returns:
        xarray.Dataset: Concatenated and filtered dataset.
    """
    files = sorted(glob.glob(path_pattern))[-6:-2]
    ds_list = []

    for f in files:
        ds = xr.open_dataset(f).isel(ATMLAY3_3=0, SECTOR=0, SECTOR1_1=0)[var_names]
        ds = ds.sel(TIME=ds['TIME'].dt.month.isin(months))
        ds['TIME'] = ds['TIME'].dt.floor('D')
        ds_list.append(ds)

    ds_combined = xr.concat(ds_list, dim='TIME').sel(TIME=slice(*time_slice))
    return ds_combined


def read_mar_var_AP(path_pattern, var_names, months=[11, 12, 1, 2, 3], time_slice=('19800101', '20220331')):
    """
    Reads MAR NetCDF files, selects specified variables and dimensions,
    filters by months and time range.

    Parameters:
        path_pattern (str): File path pattern for NetCDF files.
        var_names (list): List of variable names to extract.
        months (list, optional): Months to filter. Default is summer in Patagonia.
        time_slice (tuple, optional): Start and end date for filtering. Default is ('19800101', '20220331').

    Returns:
        xarray.Dataset: Concatenated and filtered dataset.
    """
    files = sorted(glob.glob(path_pattern))[-4:]
    ds_list = []

    for f in files:
        ds = xr.open_dataset(f).sel(X = slice(-2660, -2120), Y= slice(800, 1700)).isel(SECTOR=0, SECTOR1_1=0, ATMLAY=0)[var_names]
        ds = ds.sel(TIME=ds['TIME'].dt.month.isin(months))
        ds['TIME'] = ds['TIME'].dt.floor('D')
        ds_list.append(ds)

    ds_combined = xr.concat(ds_list, dim='TIME').sel(TIME=slice(*time_slice))
    return ds_combined

def compute_daily_anomalies(ds: xr.DataArray, time_dim: str = "time") -> xr.DataArray:
    """
    Compute daily anomalies of a temperature field based on day-of-year climatology.
    
    Parameters:
        ds (xr.DataArray): Input data with a time dimension (e.g., daily temperature).
        time_dim (str): Name of the time dimension (default is 'time').

    Returns:
        anomalies (xr.DataArray): Daily anomalies with same shape as input.
    """

    # Create dayofyear coordinate
    doy = ds[time_dim].dt.dayofyear
    ds = ds.assign_coords(dayofyear=(time_dim, doy.data))

    # Compute climatology (mean for each day of the year)
    climatology = ds.groupby("dayofyear").mean(time_dim)

    # Subtract climatology to get anomalies
    anomalies = ds.groupby("dayofyear") - climatology

    return anomalies


In [3]:
files = sorted(glob.glob('/media/lacrio/TOSHIBA EXT/Postdoc_project/MAR_DATA_SEL/Patagonia/*.nc'))
ds_MAR = xr.open_dataset(files[0])
LAT = ds_MAR['LAT']
LON = ds_MAR['LON']

files = sorted(glob.glob('/media/lacrio/TOSHIBA EXT/Postdoc_project/MAR_DATA_SEL/APeninsula/*.nc'))
ds_MAR_AP = xr.open_dataset(files[0]).sel(X = slice(-2660, -2120), Y= slice(800, 1700))
LAT_AP = ds_MAR_AP['LAT']
LON_AP = ds_MAR_AP['LON']

In [4]:
with open('/home/lacrio/Postdoc_project/MAR_data/mask_PAT.npy', 'rb') as f:
    mask_PAT = np.load(f)

In [5]:
import cartopy.feature as cfeature
from shapely.geometry import Point
from shapely.prepared import prep

# --- Coastline polygons from Natural Earth (land) ---
land_feature = cfeature.NaturalEarthFeature("physical", "land", "10m")
land_geom = [prep(geom) for geom in land_feature.geometries()]

# --- Create coordinate grids ---
lon2d, lat2d = LON_AP, LAT_AP
mask_AP = np.zeros(lon2d.shape, dtype=bool)

# --- Build mask: True = continente
for i in range(lon2d.shape[0]):
    for j in range(lon2d.shape[1]):
        pt = Point(lon2d[i, j], lat2d[i, j])
        # True se está na terra
        if any(geom.contains(pt) for geom in land_geom):
            mask_AP[i, j] = True
        else:
            mask_AP[i, j] = False


# --- 🔧 Apply latitude limits (-68 ≤ lat ≤ -60)
#mask_AP[(lat2d < -68) | (lat2d > -60)] = False

In [6]:
import xarray as xr


def process_mar_patagonia_2d(path_pattern, mask):
    """Procesa SEB en 2D para Patagonia usando SWU directo e incluye SWnet, LWnet y Qnet."""
    ds_SWD = read_mar_var(path_pattern=path_pattern, var_names=['SWD']).where(mask)
    ds_SWU = read_mar_var(path_pattern=path_pattern, var_names=['SWU']).where(mask)
    ds_LWD = read_mar_var(path_pattern=path_pattern, var_names=['LWD']).where(mask)
    ds_LWU = read_mar_var(path_pattern=path_pattern, var_names=['LWU']).where(mask)
    ds_SHF = read_mar_var(path_pattern=path_pattern, var_names=['SHF']).where(mask)
    ds_LHF = read_mar_var(path_pattern=path_pattern, var_names=['LHF']).where(mask)

    # Componentes individuales (en 2D espacial)
    swd = ds_SWD['SWD']
    swu = -1 * ds_SWU['SWU']
    lwd = ds_LWD['LWD']
    lwu = -1 * ds_LWU['LWU']
    shf = ds_SHF['SHF']
    lhf = ds_LHF['LHF']

    # Componentes NETAS (2D)
    swnet = ds_SWD['SWD'] - ds_SWU['SWU']
    lwnet = ds_LWD['LWD'] - ds_LWU['LWU']

    # Balance Neto Total Qnet (2D)
    qnet = swnet + lwnet + shf + lhf

    # Retornar como un Dataset de Xarray
    return xr.Dataset(
        {
            "SWD": swd,
            "SWU": swu,
            "LWD": lwd,
            "LWU": lwu,
            "SWnet": swnet,
            "LWnet": lwnet,
            "SHF": shf,
            "LHF": lhf,
            "Qnet": qnet,
        }
    )


def process_mar_apis_2d(path_pattern, mask):
    """Procesa SEB en 2D para Península Antártica usando Albedo e incluye SWnet, LWnet y Qnet."""
    ds_SWD = read_mar_var_AP(path_pattern=path_pattern, var_names=['SWD']).where(mask)
    ds_ALB = read_mar_var_AP(path_pattern=path_pattern, var_names=['AL']).where(mask)
    ds_LWD = read_mar_var_AP(path_pattern=path_pattern, var_names=['LWD']).where(mask)
    ds_LWU = read_mar_var_AP(path_pattern=path_pattern, var_names=['LWU']).where(mask)
    ds_SHF = read_mar_var_AP(path_pattern=path_pattern, var_names=['SHF']).where(mask)
    ds_LHF = read_mar_var_AP(path_pattern=path_pattern, var_names=['LHF']).where(mask)

    # Componentes individuales (en 2D espacial)
    swd = ds_SWD['SWD']
    swu = -1 * (ds_SWD['SWD'] * ds_ALB['AL'])
    lwd = ds_LWD['LWD']
    lwu = -1 * ds_LWU['LWU']
    shf = ds_SHF['SHF']
    lhf = ds_LHF['LHF']

    # Componentes NETAS considerando Albedo (2D)
    swnet = ds_SWD['SWD'] * (1 - ds_ALB['AL'])
    lwnet = ds_LWD['LWD'] - ds_LWU['LWU']

    # Balance Neto Total Qnet (2D)
    qnet = swnet + lwnet + shf + lhf

    # Retornar como un Dataset de Xarray
    return xr.Dataset(
        {
            "SWD": swd,
            "SWU": swu,
            "LWD": lwd,
            "LWU": lwu,
            "SWnet": swnet,
            "LWnet": lwnet,
            "SHF": shf,
            "LHF": lhf,
            "Qnet": qnet,
        }
    )

In [7]:
# =============================================================================
# 2. CARGA DE DATOS
# =============================================================================
path_patagonia = '/media/lacrio/TOSHIBA EXT/Postdoc_project/MAR_DATA_SEL/Patagonia/*.nc'
path_peninsula = '/media/lacrio/TOSHIBA EXT/Postdoc_project/MAR_DATA_SEL/APeninsula/*.nc'

# ============================================================
# 1. EJECUCIÓN DE LAS FUNCIONES 2D
# ============================================================
ds_PIs = process_mar_patagonia_2d(path_patagonia, mask_PAT)
ds_APIS = process_mar_apis_2d(path_peninsula, mask_AP)


In [9]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import Reader as ShapeReader
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# ------------------------------------------------------------
# 1. CONFIGURACIÓN Y PARÁMETROS DE MAPA
# ------------------------------------------------------------
proj_map  = ccrs.PlateCarree()
proj_data = ccrs.PlateCarree()

extent = [-74.9, -71.80, -51.75, -46.25]
xticks = np.arange(-74.3, -72.0, 1)
yticks = np.arange(-51.5, -46.0, 1)

shp_path = "/home/lacrio/Patagonia-Antarctic_Peninsula/data/shp/Glaciers_NPI_SPI.shp"

# Rango único de colores
cmap_common = "RdBu_r"
vmin_common = -200
vmax_common = 200

date_ranges = [
    {
        "start": np.datetime64("2019-01-31"),
        "end": np.datetime64("2019-02-08"),
        "event_start": np.datetime64("2019-02-04"),
        "event_end": np.datetime64("2019-02-04"),
        "label": "2019",
    },
    {
        "start": np.datetime64("2020-02-05"),
        "end": np.datetime64("2020-02-13"),
        "event_start": np.datetime64("2020-02-10"),
        "event_end": np.datetime64("2020-02-10"),
        "label": "2020",
    },
    {
        "start": np.datetime64("2021-02-06"),
        "end": np.datetime64("2021-02-14"),
        "event_start": np.datetime64("2021-02-09"),
        "event_end": np.datetime64("2021-02-11"),
        "label": "2021",
    },
    {
        "start": np.datetime64("2022-02-03"),
        "end": np.datetime64("2022-02-11"),
        "event_start": np.datetime64("2022-02-07"),
        "event_end": np.datetime64("2022-02-07"),
        "label": "2022",
    },
]

# ------------------------------------------------------------
# 2. FUNCIÓN AUXILIAR PARA PANEL DE MAPA (SIN BARRA INDIVIDUAL)
# ------------------------------------------------------------
def add_panel(ax, data, lon, lat, cmap, vmin, vmax, letter, title):
    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")
    ax.add_feature(cfeature.LAKES, facecolor="lightblue", edgecolor="black")

    ax.add_geometries(
        ShapeReader(shp_path).geometries(),
        facecolor="none", edgecolor="k", lw=0.4,
        crs=proj_data
    )

    ax.set_extent(extent, crs=proj_data)
    ax.set_xticks(xticks, crs=proj_data)
    ax.set_yticks(yticks, crs=proj_data)

    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(axis="both", labelsize=9)

    mesh = ax.pcolormesh(
        lon, lat, data,
        cmap=cmap, vmin=vmin, vmax=vmax,
        shading="auto", transform=proj_data
    )

    ax.text(0.04, 0.98, f"{letter} {title}", transform=ax.transAxes,
            fontsize=10, fontweight="bold",
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
    
    return mesh

# ------------------------------------------------------------
# 3. PROCESAMIENTO Y PLOTEO POR EVENTO (ESTRUCTURA 2x3)
# ------------------------------------------------------------
ds_pat_base = ds_PIs

os.makedirs("fig", exist_ok=True)

for event in date_ranges:
    label = event["label"]
    t_start = event["event_start"]
    t_end = event["event_end"]
    
    print(f"\n--- Procesando y graficando Evento {label} ({t_start} a {t_end}) ---")

    # 1. Filtrar temporalmente el evento
    ds_evt = ds_pat_base.sel(TIME=slice(t_start, t_end))

    # 2. Promedio condicional si dura más de 1 día
    duration_days = (t_end - t_start).astype("timedelta64[D]").astype(int) + 1

    if duration_days > 1:
        print(f"Evento de {duration_days} días: calculando media temporal .mean('TIME')...")
        ds_final = ds_evt.mean(dim="TIME")
    else:
        print("Evento de 1 día: manteniendo el estado diario sin promediar...")
        ds_final = ds_evt.squeeze(dim="TIME", drop=True) if "TIME" in ds_evt.dims else ds_evt

    # 3. Preparar lista de variables a graficar
    panels_data = [
        (ds_final["SWnet"], "SWnet"),
        (ds_final["LWnet"], "LWnet"),
        (ds_final["SHF"], "SHF"),
        (ds_final["LHF"], "LHF"),
        (ds_final["Qnet"], "SEB")
    ]
    
    letters = ["(a)", "(b)", "(c)", "(d)", "(e)"]

    # 4. Crear figura 2x3 (Ancho=15, Alto=9)
    fig = plt.figure(figsize=(15, 15))
    gs = gridspec.GridSpec(nrows=2, ncols=3, figure=fig, wspace=0.2, hspace=0.25)

    mesh_last = None

    time_info = f"{t_start} a {t_end}" if duration_days > 1 else f"{t_start}"
    fig.suptitle(f"Surface Energy Balance - PIs Event {label} ({time_info})", 
                 fontsize=14, fontweight="bold", y=0.90)

    # Plotear las 5 variables en la grilla 2x3
    for i, (da, title) in enumerate(panels_data):
        row = i // 3
        col = i % 3
        
        ax = fig.add_subplot(gs[row, col], projection=proj_map)
        mesh_last = add_panel(
            ax, da, LON, LAT,
            cmap=cmap_common, vmin=vmin_common, vmax=vmax_common,
            letter=letters[i], title=title
        )

# ------------------------------------------------------------
    # 5. Posición (1, 2) / Panel 6: BARRA DE COLOR ÚNICA CENTRADA
    # ------------------------------------------------------------
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    ax_cbar = fig.add_subplot(gs[1, 2])
    ax_cbar.axis("off")  # Ocultar ejes del sexto panel

    # Crear un eje "inset" centrado exactamente dentro del sexto subplot
    # width="12%" controla el grosor de la barra, height="85%" la ajusta a la altura del mapa
    cax = inset_axes(
        ax_cbar,
        width="12%",
        height="85%",
        loc="center"
    )

    cbar = fig.colorbar(
        mesh_last,
        cax=cax,
        orientation="vertical",
        extend="both"
    )

    cbar.set_label("Energy Flux (W m$^{-2}$)", fontsize=11, fontweight="bold")
    cbar.ax.tick_params(labelsize=10)

    # 6. Guardar la figura del evento
    output_pdf = f"fig/Fig_SEB_2x3_Event_{label}.png"
    fig.savefig(
        output_pdf,
        dpi=300, facecolor="w",
        bbox_inches="tight", pad_inches=0.1
    )
    plt.close(fig)
    print(f"--> Guardado con éxito: {output_pdf}")


--- Procesando y graficando Evento 2019 (2019-02-04 a 2019-02-04) ---
Evento de 1 día: manteniendo el estado diario sin promediar...


/home/lacrio/miniconda3/envs/MassBalanceMachine/lib/python3.11/site-packages/cartopy/io/__init__.py:241: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/10m_physical/ne_10m_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


--> Guardado con éxito: fig/Fig_SEB_2x3_Event_2019.png

--- Procesando y graficando Evento 2020 (2020-02-10 a 2020-02-10) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SEB_2x3_Event_2020.png

--- Procesando y graficando Evento 2021 (2021-02-09 a 2021-02-11) ---
Evento de 3 días: calculando media temporal .mean('TIME')...
--> Guardado con éxito: fig/Fig_SEB_2x3_Event_2021.png

--- Procesando y graficando Evento 2022 (2022-02-07 a 2022-02-07) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SEB_2x3_Event_2022.png


In [10]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# ------------------------------------------------------------
# 1. CONFIGURACIÓN Y PARÁMETROS DE MAPA (APIS)
# ------------------------------------------------------------
proj_map  = ccrs.Mercator()
proj_data = ccrs.PlateCarree()

extent = [-70, -55.5, -69.4, -61.0]  # [lon_min, lon_max, lat_min, lat_max]
xticks = np.arange(-68.7, -56, 4)
yticks = np.arange(-69, -61.0, 1.5)

# Rango único de colores
cmap_common = "RdBu_r"
vmin_common = -200
vmax_common = 200

date_ranges = [
    {
        "start": np.datetime64("2019-01-31"),
        "end": np.datetime64("2019-02-08"),
        "event_start": np.datetime64("2019-02-04"),
        "event_end": np.datetime64("2019-02-04"),
        "label": "2019",
    },
    {
        "start": np.datetime64("2020-02-05"),
        "end": np.datetime64("2020-02-13"),
        "event_start": np.datetime64("2020-02-10"),
        "event_end": np.datetime64("2020-02-10"),
        "label": "2020",
    },
    {
        "start": np.datetime64("2021-02-06"),
        "end": np.datetime64("2021-02-14"),
        "event_start": np.datetime64("2021-02-09"),
        "event_end": np.datetime64("2021-02-11"),
        "label": "2021",
    },
    {
        "start": np.datetime64("2022-02-03"),
        "end": np.datetime64("2022-02-11"),
        "event_start": np.datetime64("2022-02-07"),
        "event_end": np.datetime64("2022-02-07"),
        "label": "2022",
    },
]

# ------------------------------------------------------------
# 2. FUNCIÓN AUXILIAR PARA PANEL DE MAPA (APIS)
# ------------------------------------------------------------
def add_panel(ax, data, lon, lat, cmap, vmin, vmax, letter, title):
    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")

    ax.set_extent(extent, crs=proj_data)
    ax.set_xticks(xticks, crs=proj_data)
    ax.set_yticks(yticks, crs=proj_data)

    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(axis="both", labelsize=9)

    mesh = ax.pcolormesh(
        lon, lat, data,
        cmap=cmap, vmin=vmin, vmax=vmax,
        shading="auto", transform=proj_data
    )

    ax.text(0.04, 0.98, f"{letter} {title}", transform=ax.transAxes,
            fontsize=10, fontweight="bold",
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
    
    return mesh

# ------------------------------------------------------------
# 3. PROCESAMIENTO Y PLOTEO POR EVENTO (ESTRUCTURA 2x3 - APIS)
# ------------------------------------------------------------
ds_apis_base = ds_APIS

os.makedirs("fig", exist_ok=True)

for event in date_ranges:
    label = event["label"]
    t_start = event["event_start"]
    t_end = event["event_end"]
    
    print(f"\n--- Procesando y graficando APIS Evento {label} ({t_start} a {t_end}) ---")

    # 1. Filtrar temporalmente el evento
    ds_evt = ds_apis_base.sel(TIME=slice(t_start, t_end))

    # 2. Promedio condicional si dura más de 1 día
    duration_days = (t_end - t_start).astype("timedelta64[D]").astype(int) + 1

    if duration_days > 1:
        print(f"Evento de {duration_days} días: calculando media temporal .mean('TIME')...")
        ds_final = ds_evt.mean(dim="TIME")
    else:
        print("Evento de 1 día: manteniendo el estado diario sin promediar...")
        ds_final = ds_evt.squeeze(dim="TIME", drop=True) if "TIME" in ds_evt.dims else ds_evt

    # 3. Preparar lista de variables a graficar con su máscara
    panels_data = [
        (ds_final["SWnet"].where(mask_AP), "SWnet"),
        (ds_final["LWnet"].where(mask_AP), "LWnet"),
        (ds_final["SHF"].where(mask_AP), "SHF"),
        (ds_final["LHF"].where(mask_AP), "LHF"),
        (ds_final["Qnet"].where(mask_AP), "SEB")
    ]
    
    letters = ["(a)", "(b)", "(c)", "(d)", "(e)"]

    # 4. Crear figura 2x3
    fig = plt.figure(figsize=(15, 14))
    gs = gridspec.GridSpec(nrows=2, ncols=3, figure=fig, wspace=0.2, hspace=0.25)

    mesh_last = None

    time_info = f"{t_start} a {t_end}" if duration_days > 1 else f"{t_start}"
    fig.suptitle(f"Surface Energy Balance - APIS Event {label} ({time_info})", 
                 fontsize=14, fontweight="bold", y=0.90)

    # Plotear las 5 variables en la grilla 2x3
    for i, (da, title) in enumerate(panels_data):
        row = i // 3
        col = i % 3
        
        ax = fig.add_subplot(gs[row, col], projection=proj_map)
        mesh_last = add_panel(
            ax, da, LON_AP, LAT_AP,
            cmap=cmap_common, vmin=vmin_common, vmax=vmax_common,
            letter=letters[i], title=title
        )

    # ------------------------------------------------------------
    # 5. Posición (1, 2) / Panel 6: BARRA DE COLOR ÚNICA CENTRADA
    # ------------------------------------------------------------
    ax_cbar = fig.add_subplot(gs[1, 2])
    ax_cbar.axis("off")  # Ocultar ejes del sexto panel

    cax = inset_axes(
        ax_cbar,
        width="12%",
        height="85%",
        loc="center"
    )

    cbar = fig.colorbar(
        mesh_last,
        cax=cax,
        orientation="vertical",
        extend="both"
    )

    cbar.set_label("Energy Flux (W m$^{-2}$)", fontsize=11, fontweight="bold")
    cbar.ax.tick_params(labelsize=10)

    # 6. Guardar la figura del evento
    output_png = f"fig/Fig_SEB_APIS_2x3_Event_{label}.png"
    fig.savefig(
        output_png,
        dpi=300, facecolor="w",
        bbox_inches="tight", pad_inches=0.1
    )
    plt.close(fig)
    print(f"--> Guardado con éxito: {output_png}")


--- Procesando y graficando APIS Evento 2019 (2019-02-04 a 2019-02-04) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SEB_APIS_2x3_Event_2019.png

--- Procesando y graficando APIS Evento 2020 (2020-02-10 a 2020-02-10) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SEB_APIS_2x3_Event_2020.png

--- Procesando y graficando APIS Evento 2021 (2021-02-09 a 2021-02-11) ---
Evento de 3 días: calculando media temporal .mean('TIME')...
--> Guardado con éxito: fig/Fig_SEB_APIS_2x3_Event_2021.png

--- Procesando y graficando APIS Evento 2022 (2022-02-07 a 2022-02-07) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SEB_APIS_2x3_Event_2022.png


In [ ]:
import xarray as xr


def process_mar_patagonia_smb_2d(path_pattern, mask):
    """Procesa en 2D las componentes del SMB para Patagonia (PR, SU, RU y SMB)."""
    ds_SF = read_mar_var(path_pattern=path_pattern, var_names=['SF']).where(mask)
    ds_RF = read_mar_var(path_pattern=path_pattern, var_names=['RF']).where(mask)
    ds_SU = read_mar_var(path_pattern=path_pattern, var_names=['SU']).where(mask)
    ds_RU = read_mar_var(path_pattern=path_pattern, var_names=['RU']).where(mask)
    ds_ME  = read_mar_var(path_pattern=path_pattern, var_names=['ME']).where(mask)

    # Precipitación total PR = SF + RF (2D)
    ds_PR = ds_SF['SF'] + ds_RF['RF']

    ds_RE = (ds_ME['ME'] + ds_RF['RF'] - ds_RU['RU']).clip(min=0) # 

    ds_SMB = ds_PR + ds_RE - ds_ME['ME'] - ds_RF['RF'] - ds_SU['SU']

    # Salidas de masa (se consideran negativas en el balance)
    me = -1 * ds_ME['ME']
    rf = -1 * ds_RF['RF']
    su = -1 * ds_SU['SU']
    ru = -1 * ds_RU['RU']

    return xr.Dataset(
        {
            "PR": ds_PR,
            "ME": me,
            "RF": rf,
            "RZ": ds_RE,
            "SU": su,
            "RU": ru,
            "SMB": ds_SMB,
        }
    )


def process_mar_apis_smb_2d(path_pattern, mask):
    """Procesa en 2D las componentes del SMB para la Península Antártica (PR, SU, RU y SMB)."""
    ds_SF = read_mar_var_AP(path_pattern=path_pattern, var_names=['SF']).where(mask)
    ds_RF = read_mar_var_AP(path_pattern=path_pattern, var_names=['RF']).where(mask)
    ds_SU = read_mar_var_AP(path_pattern=path_pattern, var_names=['SU']).where(mask)
    ds_RU = read_mar_var_AP(path_pattern=path_pattern, var_names=['RU']).where(mask)
    ds_ME = read_mar_var_AP(path_pattern=path_pattern, var_names=['ME']).where(mask)

    # Precipitación total PR = SF + RF (2D)
    ds_PR = ds_SF['SF'] + ds_RF['RF']

    ds_RE = (ds_ME['ME'] + ds_RF['RF'] - ds_RU['RU']).clip(min=0) # 

    ds_SMB = ds_PR + ds_RE - ds_ME['ME'] - ds_RF['RF'] - ds_SU['SU']

    # Salidas de masa (se consideran negativas en el balance)
    me = -1 * ds_ME['ME']
    rf = -1 * ds_RF['RF']
    su = -1 * ds_SU['SU']
    ru = -1 * ds_RU['RU']


    return xr.Dataset(
        {
            "PR": ds_PR,
            "ME": me,
            "RF": rf,
            "RZ": ds_RE,
            "SU": su,
            "RU": ru,
            "SMB": ds_SMB,
        }
    )


In [14]:
# =============================================================================
# 2. CARGA DE DATOS
# =============================================================================
path_patagonia = '/media/lacrio/TOSHIBA EXT/Postdoc_project/MAR_DATA_SEL/Patagonia/*.nc'
path_peninsula = '/media/lacrio/TOSHIBA EXT/Postdoc_project/MAR_DATA_SEL/APeninsula/*.nc'

# Ejecución de funciones
# Cargar Datasets 2D
ds_smb_PIs = process_mar_patagonia_smb_2d(path_patagonia, mask_PAT)
ds_smb_APIS = process_mar_apis_smb_2d(path_peninsula, mask_AP)

In [17]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io.shapereader import Reader as ShapeReader
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# ------------------------------------------------------------
# 1. CONFIGURACIÓN Y PARÁMETROS DE MAPA
# ------------------------------------------------------------
proj_map  = ccrs.PlateCarree()
proj_data = ccrs.PlateCarree()

extent = [-74.9, -71.80, -51.75, -46.25]
xticks = np.arange(-74.3, -72.0, 1)
yticks = np.arange(-51.5, -46.0, 1)

shp_path = "/home/lacrio/Patagonia-Antarctic_Peninsula/data/shp/Glaciers_NPI_SPI.shp"

# Configuración de paleta y rangos de colores para SMB (mmWE d^-1)
cmap_common = "RdBu"
vmin_common = -80
vmax_common = 80

date_ranges = [
    {
        "start": np.datetime64("2019-01-31"),
        "end": np.datetime64("2019-02-08"),
        "event_start": np.datetime64("2019-02-04"),
        "event_end": np.datetime64("2019-02-04"),
        "label": "2019",
    },
    {
        "start": np.datetime64("2020-02-05"),
        "end": np.datetime64("2020-02-13"),
        "event_start": np.datetime64("2020-02-10"),
        "event_end": np.datetime64("2020-02-10"),
        "label": "2020",
    },
    {
        "start": np.datetime64("2021-02-06"),
        "end": np.datetime64("2021-02-14"),
        "event_start": np.datetime64("2021-02-09"),
        "event_end": np.datetime64("2021-02-11"),
        "label": "2021",
    },
    {
        "start": np.datetime64("2022-02-03"),
        "end": np.datetime64("2022-02-11"),
        "event_start": np.datetime64("2022-02-07"),
        "event_end": np.datetime64("2022-02-07"),
        "label": "2022",
    },
]

# ------------------------------------------------------------
# 2. FUNCIÓN AUXILIAR PARA PANEL DE MAPA
# ------------------------------------------------------------
def add_panel(ax, data, lon, lat, cmap, vmin, vmax, letter, title):
    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")
    ax.add_feature(cfeature.LAKES, facecolor="lightblue", edgecolor="black")

    ax.add_geometries(
        ShapeReader(shp_path).geometries(),
        facecolor="none", edgecolor="k", lw=0.4,
        crs=proj_data
    )

    ax.set_extent(extent, crs=proj_data)
    ax.set_xticks(xticks, crs=proj_data)
    ax.set_yticks(yticks, crs=proj_data)

    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(axis="both", labelsize=8)

    mesh = ax.pcolormesh(
        lon, lat, data,
        cmap=cmap, vmin=vmin, vmax=vmax,
        shading="auto", transform=proj_data
    )

    ax.text(0.04, 0.98, f"{letter} {title}", transform=ax.transAxes,
            fontsize=9, fontweight="bold",
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
    
    return mesh

# ------------------------------------------------------------
# 3. PROCESAMIENTO Y PLOTEO POR EVENTO (ESTRUCTURA 2 Filas x 3 Columnas)
# ------------------------------------------------------------
# Cargar datos base 2D de SMB (ej. ds_smb_PIs)
ds_smb_base = ds_smb_PIs

os.makedirs("fig", exist_ok=True)

for event in date_ranges:
    label = event["label"]
    t_start = event["event_start"]
    t_end = event["event_end"]
    
    print(f"\n--- Procesando y graficando Evento SMB {label} ({t_start} a {t_end}) ---")

    # 1. Filtrar temporalmente el evento
    ds_evt = ds_smb_base.sel(TIME=slice(t_start, t_end))

    # 2. Promedio condicional si dura más de 1 día
    duration_days = (t_end - t_start).astype("timedelta64[D]").astype(int) + 1

    if duration_days > 1:
        print(f"Evento de {duration_days} días: calculando media temporal .mean('TIME')...")
        ds_final = ds_evt.mean(dim="TIME")
    else:
        print("Evento de 1 día: manteniendo el estado diario sin promediar...")
        ds_final = ds_evt.squeeze(dim="TIME", drop=True) if "TIME" in ds_evt.dims else ds_evt

    # 3. Preparar las 6 variables de SMB
    panels_data = [
        (ds_final["PR"].where(mask_PAT), "Precipitation"),
        (ds_final["ME"].where(mask_PAT), "Melt"),
        (ds_final["RF"].where(mask_PAT), "Rainfall"),
        (ds_final["RZ"].where(mask_PAT), "Refreezing"),
        (ds_final["SU"].where(mask_PAT), "Sublimation"),
        (ds_final["SMB"].where(mask_PAT), "Surface Mass Balance"),
    ]
    
    letters = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    # 4. Crear figura de 2 filas x 3 columnas
    fig = plt.figure(figsize=(15, 15))
    gs = gridspec.GridSpec(nrows=2, ncols=3, figure=fig, wspace=0.15, hspace=0.2)

    mesh_last = None

    time_info = f"{t_start} a {t_end}" if duration_days > 1 else f"{t_start}"
    fig.suptitle(f"Surface Mass Balance - PIs Event {label} ({time_info})", 
                 fontsize=14, fontweight="bold", y=0.90)

    # Plotear los 6 mapas en la grilla 2x3
    for i, (da, title) in enumerate(panels_data):
        row = i // 3
        col = i % 3
        
        ax = fig.add_subplot(gs[row, col], projection=proj_map)
        mesh_last = add_panel(
            ax, da, LON, LAT,
            cmap=cmap_common, vmin=vmin_common, vmax=vmax_common,
            letter=letters[i], title=title
        )

    # ------------------------------------------------------------
    # 5. BARRA DE COLOR ÚNICA Y COMPARTIDA A LA DERECHA
    # ------------------------------------------------------------
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.20, 0.02, 0.60])  # [left, bottom, width, height]

    cbar = fig.colorbar(
        mesh_last,
        cax=cbar_ax,
        orientation="vertical",
        extend="both"
    )

    cbar.set_label("Mass Fluxes (mm WE day$^{-1}$)", fontsize=11, fontweight="bold")
    cbar.ax.tick_params(labelsize=10)

    # 6. Guardar la figura del evento
    output_png = f"fig/Fig_SMB_3x2_Event_{label}.png"
    fig.savefig(
        output_png,
        dpi=300, facecolor="w",
        bbox_inches="tight", pad_inches=0.1
    )
    plt.close(fig)
    print(f"--> Guardado con éxito: {output_png}")


--- Procesando y graficando Evento SMB 2019 (2019-02-04 a 2019-02-04) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SMB_3x2_Event_2019.png

--- Procesando y graficando Evento SMB 2020 (2020-02-10 a 2020-02-10) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SMB_3x2_Event_2020.png

--- Procesando y graficando Evento SMB 2021 (2021-02-09 a 2021-02-11) ---
Evento de 3 días: calculando media temporal .mean('TIME')...
--> Guardado con éxito: fig/Fig_SMB_3x2_Event_2021.png

--- Procesando y graficando Evento SMB 2022 (2022-02-07 a 2022-02-07) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SMB_3x2_Event_2022.png


In [18]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# ------------------------------------------------------------
# 1. CONFIGURACIÓN Y PARÁMETROS DE MAPA (APIS)
# ------------------------------------------------------------
proj_map  = ccrs.Mercator()
proj_data = ccrs.PlateCarree()

extent = [-70, -55.5, -69.4, -61.0]  # [lon_min, lon_max, lat_min, lat_max]
xticks = np.arange(-68.7, -56, 4)
yticks = np.arange(-69, -61.0, 1.5)

# Configuración de colores para SMB
cmap_common = "RdBu"
vmin_common = -80
vmax_common = 80

date_ranges = [
    {
        "start": np.datetime64("2019-01-31"),
        "end": np.datetime64("2019-02-08"),
        "event_start": np.datetime64("2019-02-04"),
        "event_end": np.datetime64("2019-02-04"),
        "label": "2019",
    },
    {
        "start": np.datetime64("2020-02-05"),
        "end": np.datetime64("2020-02-13"),
        "event_start": np.datetime64("2020-02-10"),
        "event_end": np.datetime64("2020-02-10"),
        "label": "2020",
    },
    {
        "start": np.datetime64("2021-02-06"),
        "end": np.datetime64("2021-02-14"),
        "event_start": np.datetime64("2021-02-09"),
        "event_end": np.datetime64("2021-02-11"),
        "label": "2021",
    },
    {
        "start": np.datetime64("2022-02-03"),
        "end": np.datetime64("2022-02-11"),
        "event_start": np.datetime64("2022-02-07"),
        "event_end": np.datetime64("2022-02-07"),
        "label": "2022",
    },
]

# ------------------------------------------------------------
# 2. FUNCIÓN AUXILIAR PARA PANEL DE MAPA (APIS)
# ------------------------------------------------------------
def add_panel(ax, data, lon, lat, cmap, vmin, vmax, letter, title):
    ax.coastlines(resolution="10m")
    ax.add_feature(cfeature.BORDERS, linestyle=":")
    ax.add_feature(cfeature.LAND, facecolor="lightgray")
    ax.add_feature(cfeature.OCEAN, facecolor="lightblue")

    ax.set_extent(extent, crs=proj_data)
    ax.set_xticks(xticks, crs=proj_data)
    ax.set_yticks(yticks, crs=proj_data)

    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(axis="both", labelsize=8)

    mesh = ax.pcolormesh(
        lon, lat, data,
        cmap=cmap, vmin=vmin, vmax=vmax,
        shading="auto", transform=proj_data
    )

    ax.text(0.04, 0.98, f"{letter} {title}", transform=ax.transAxes,
            fontsize=9, fontweight="bold",
            ha="left", va="top",
            bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))
    
    return mesh

# ------------------------------------------------------------
# 3. PROCESAMIENTO Y PLOTEO POR EVENTO (ESTRUCTURA 2 Filas x 3 Columnas - APIS)
# ------------------------------------------------------------
ds_smb_apis_base = ds_smb_APIS

os.makedirs("fig", exist_ok=True)

for event in date_ranges:
    label = event["label"]
    t_start = event["event_start"]
    t_end = event["event_end"]
    
    print(f"\n--- Procesando y graficando APIS Evento SMB {label} ({t_start} a {t_end}) ---")

    # 1. Filtrar temporalmente el evento
    ds_evt = ds_smb_apis_base.sel(TIME=slice(t_start, t_end))

    # 2. Promedio condicional si dura más de 1 día
    duration_days = (t_end - t_start).astype("timedelta64[D]").astype(int) + 1

    if duration_days > 1:
        print(f"Evento de {duration_days} días: calculando media temporal .mean('TIME')...")
        ds_final = ds_evt.mean(dim="TIME")
    else:
        print("Evento de 1 día: manteniendo el estado diario sin promediar...")
        ds_final = ds_evt.squeeze(dim="TIME", drop=True) if "TIME" in ds_evt.dims else ds_evt

    # 3. Preparar las 6 variables de SMB aplicando la máscara de APIS
    panels_data = [
        (ds_final["PR"].where(mask_AP), "Precipitation"),
        (ds_final["ME"].where(mask_AP), "Melt"),
        (ds_final["RF"].where(mask_AP), "Rainfall"),
        (ds_final["RZ"].where(mask_AP), "Refreezing"),
        (ds_final["SU"].where(mask_AP), "Sublimation"),
        (ds_final["SMB"].where(mask_AP), "Surface Mass Balance"),
    ]
    
    letters = ["(a)", "(b)", "(c)", "(d)", "(e)", "(f)"]

    # 4. Crear figura de 2 filas x 3 columnas
    fig = plt.figure(figsize=(15, 14))
    gs = gridspec.GridSpec(nrows=2, ncols=3, figure=fig, wspace=0.15, hspace=0.2)

    mesh_last = None

    time_info = f"{t_start} a {t_end}" if duration_days > 1 else f"{t_start}"
    fig.suptitle(f"Surface Mass Balance - APIS Event {label} ({time_info})", 
                 fontsize=14, fontweight="bold", y=0.90)

    # Plotear los 6 mapas en la grilla 2x3
    for i, (da, title) in enumerate(panels_data):
        row = i // 3
        col = i % 3
        
        ax = fig.add_subplot(gs[row, col], projection=proj_map)
        mesh_last = add_panel(
            ax, da, LON_AP, LAT_AP,
            cmap=cmap_common, vmin=vmin_common, vmax=vmax_common,
            letter=letters[i], title=title
        )

    # ------------------------------------------------------------
    # 5. BARRA DE COLOR ÚNICA Y COMPARTIDA A LA DERECHA
    # ------------------------------------------------------------
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.20, 0.02, 0.60])  # [left, bottom, width, height]

    cbar = fig.colorbar(
        mesh_last,
        cax=cbar_ax,
        orientation="vertical",
        extend="both"
    )

    cbar.set_label("Mass Fluxes (mm WE day$^{-1}$)", fontsize=11, fontweight="bold")
    cbar.ax.tick_params(labelsize=10)

    # 6. Guardar la figura del evento
    output_png = f"fig/Fig_SMB_APIS_3x2_Event_{label}.png"
    fig.savefig(
        output_png,
        dpi=300, facecolor="w",
        bbox_inches="tight", pad_inches=0.1
    )
    plt.close(fig)
    print(f"--> Guardado con éxito: {output_png}")


--- Procesando y graficando APIS Evento SMB 2019 (2019-02-04 a 2019-02-04) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SMB_APIS_3x2_Event_2019.png

--- Procesando y graficando APIS Evento SMB 2020 (2020-02-10 a 2020-02-10) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SMB_APIS_3x2_Event_2020.png

--- Procesando y graficando APIS Evento SMB 2021 (2021-02-09 a 2021-02-11) ---
Evento de 3 días: calculando media temporal .mean('TIME')...
--> Guardado con éxito: fig/Fig_SMB_APIS_3x2_Event_2021.png

--- Procesando y graficando APIS Evento SMB 2022 (2022-02-07 a 2022-02-07) ---
Evento de 1 día: manteniendo el estado diario sin promediar...
--> Guardado con éxito: fig/Fig_SMB_APIS_3x2_Event_2022.png
